In [1]:
import pandas as pd 

df = pd.read_csv('country_info.csv')

/Users/justinwyrley/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Add duck data

In [2]:
duck = pd.read_csv('random_data/duck_data.csv', dtype={'Population Count': str, 'Ducks Per Capita (per 1.000)': str})
duck['name'] = duck['Country/Territory']

# use index as duck population ranking
duck.reset_index(inplace=True)
for i in range(len(duck)):
    duck.at[i, 'index'] = duck.at[i, 'index'] + 1

duck['duck_pop_rank'] = duck['index'].astype('Int64')

# Only keep population ranking maybe can find a use for other data
duck = duck[['duck_pop_rank', 'name']]

df = pd.merge(df, duck, on='name', how='left')

## Manually add missing into 

In [3]:
df.at[0, "calling_code"] = "+93" # Afghanistan
df.at[97, "time_zone"] = "UTC +1" # Luxembourg
df.at[20, "languages"] = "bosnian" # Bosnia and Herzegovina
df.at[108, "area_total"] = "33,843 km 2" # Moldova
df.at[29, "largest_religion"] = "Christianity" # Canada
df.at[36, "largest_religion"] = "Islam" # Comoros
df.at[61, "largest_religion"] = "Christianity" # Germany
df.at[70, "largest_religion"] = "Christianity" # Honduras
df.at[81, "largest_religion"] = "Shinto" # Japan
df.at[116, "largest_religion"] = "Christianity" # Nauru
df.at[136, "largest_religion"] = "Christianity" # Portugal
df.at[141, "largest_religion"] = "Christianity" # Russia
df.at[179, "largest_religion"] = "Islam" # Turkey
df.at[138, "time_zone"] = "UTC ±0" # Republic of Ireland


## Number rounding

In [4]:
import re

def format_population(pop_str):
    if pd.isna(pop_str): return ""
    
    # Extract numeric part
    multiplier = 1_000_000 if 'million' in pop_str.lower() else 1
    nums = re.findall(r'\d+(?:,\d+)*(?:\.\d+)?', pop_str)
    if not nums: return ""
    
    val = float(nums[0].replace(',', '')) * multiplier
    
    # Rounding logic
    if val >= 1_000_000:
        rounded_m = round(val / 1_000_000, 1)
        return f"{int(rounded_m) if rounded_m == int(rounded_m) else rounded_m} million"
    elif val >= 100_000:
        return f"{int(round(val, -5)):,}"
    elif val >= 1_000:
        return f"{int(round(val, -3)):,}"
    else:
        return str(int(val))

df['population'] = df['population'].apply(format_population)

## Alphabetical country ranking

In [5]:
# Use index to rank countries alphabetically
df.reset_index(inplace=True)
df.rename(columns={'index': 'alphabetic_country_rank'}, inplace=True)
for i in range(len(df)):
    df.at[i, 'alphabetic_country_rank'] = df.at[i, 'alphabetic_country_rank'] + 1

## Clean up timezones

In [6]:
def format_timezone(text):
    if pd.isna(text):
        return text
    
    # 1. Standardise dashes and minus signs
    text = text.replace('–', '-').replace('—', '-').replace('−', '-')
    
    # 2. Slice off anything inside or after parentheses
    text = text.split('(')[0]
    
    # 3. Clean and format the numerical offsets
    # Group 1: Optional sign (+, -, ±)
    # Group 2: Hours (\d+)
    # Group 3: Optional fraction/minutes (e.g., :30, .5, :00)
    def clean_offset(match):
        sign = match.group(1).strip() if match.group(1) else ''
        hours = int(match.group(2)) # Converts '01' to 1, '00' to 0
        fraction = match.group(3) if match.group(3) else ''
        
        # If minutes are exactly :00 or .0, drop them completely
        if fraction in [':00', '.0', '']:
            return f"{sign}{hours}"
        return f"{sign}{hours}{fraction}"
        
    text = re.sub(r'([±+-]?)\s*(\d+)((?:[:.]\d+)?)', clean_offset, text)
    
    # 4. Remove all unwanted alphabetical text
    # We keep 'UTC', 'to', and 'and' to preserve ranges and multiple zones
    words_to_keep = {'utc', 'to', 'and'}
    def filter_words(match):
        word = match.group(0)
        if word.lower() in words_to_keep:
            return 'UTC' if word.lower() == 'utc' else word.lower()
        return ''
        
    text = re.sub(r'[A-Za-z]+', filter_words, text)
    
    # 5. Tidy up spacing and ensure 'UTC' always has a trailing space
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'UTC(?=[^\s])', 'UTC ', text) 
    
    # 6. Remove any dangling punctuation at the very end
    text = text.strip(' ,;/')
    
    return text

# Apply the transformation
df['time_zone'] = df['time_zone'].apply(format_timezone)

zero_replacements = {
    'UTC +0': 'UTC ±0',
    'UTC -0': 'UTC ±0',
    'UTC 0': 'UTC ±0'
}

# Iterate through the dictionary and replace substrings directly
for old_val, new_val in zero_replacements.items():
    df['time_zone'] = df['time_zone'].str.replace(old_val, new_val, regex=False)

## Clean up area total 

In [7]:
def format_area(area_str):
    if pd.isna(area_str): return ""
    
    # 1. Target the number explicitly associated with 'km'
    # This safely ignores 'sq mi' regardless of whether it is inside or outside brackets
    match = re.search(r'([\d,]+(?:\.\d+)?)\s*km', str(area_str), re.IGNORECASE)
    
    if match:
        # Extract and clean the matched number
        num_str = match.group(1).replace(',', '')
        val = float(num_str)
        
        # Format neatly: integers get commas, floats keep their decimals
        if val.is_integer():
            return f"{int(val):,} km²"
        else:
            return f"{val:,} km²"
            
    # 2. Fallback: If 'km' is completely missing but a number exists, grab the first number
    # nums = re.findall(r'\d+(?:,\d+)*(?:\.\d+)?', str(area_str))
    # if nums:
    #     return f"{nums[0]} km²"
        
    return ""

# Apply the formatting
df['area_total'] = df['area_total'].apply(format_area)

# Clean currency

In [8]:
def format_currency(text):
    if pd.isna(text): 
        return ""
    
    # 1. Find all contents inside any brackets
    matches = re.findall(r'\(([^)]+)\)', str(text))
    
    if not matches:
        return ""
        
    formatted_items = []
    for match in matches:
        # Clean up any messy whitespace inside the brackets
        inner_text = match.strip()
        
        # 2. Check if the text consists strictly of standard letters
        # This perfectly isolates codes like 'EUR', 'USD', 'AFN'
        if re.fullmatch(r'[A-Za-z]+', inner_text):
            formatted_items.append(f"({inner_text})")
        else:
            # Anything with symbols, numbers, or non-Latin scripts (e.g. €, R$, £, ₼)
            # drops the brackets and stands alone
            formatted_items.append(inner_text)
            
    # Join the processed items together with a space
    return ' '.join(formatted_items)

# Apply the formatting to the dataset
df['currency'] = df['currency'].apply(format_currency)

In [10]:
save = df.to_csv('country_info_updated.csv', index=False)

# Clean capitals, religions then add continent

## Something to sort out later 

In [ ]:
import pandas as pd
import re

def clean_largest_religion(text):
    if pd.isna(text):
        return text
    
    # 1. Remove administrative markers
    text = re.sub(r'\(?\s*official\s*\)?', '', text, flags=re.IGNORECASE).strip()
    
    # 2. Split by percentage markers to find the primary segment
    perc_pattern = r'[~<>]?\s*\d+(?:\.\d+)?%'
    parts = re.split(perc_pattern, text)
    
    # Identify the candidate string (the one following the first percentage or the start)
    candidate = parts[1] if len(parts) > 1 and len(parts[0].strip()) <= 3 else parts[0]
    
    # 3. Strip punctuation and normalize separators
    candidate = re.sub(r'[\(\)\[\]*]', '', candidate)
    candidate = candidate.replace('—', ' ').replace('-', ' ').replace(',', ' ')
    
    words = candidate.strip().split()
    if not words: return ""
    
    #4. Handle multi-word religious names (e.g., Sunni Islam, Catholic Church)
    n = 1
    compounds = ['no', 'sunni', 'shia', 'roman', 'eastern', 'catholic', 'protestant', 'greek', 'armenian']
    if len(words) > 1:
        if words[0].lower() in compounds:
            n = 2
            # Check for 3-word names like "Armenian Apostolic Church"
            if len(words) > 2 and words[2].lower() in ['church', 'apostolic', 'orthodoxy']:
                n = 3
        elif words[1].lower() in ['islam', 'christianity', 'religion']:
            n = 2
            
    return " ".join(words[:n]).strip()

# Apply the transformation and save
df['largest_religion'] = df['largest_religion'].apply(clean_largest_religion)

In [ ]:
df[df['largest_religion'] == 'Dharmic']

,name,url,capital,languages,largest_religion,area_total,population,gdp_nominal_total,gdp_nominal_per_capita,currency,time_zone,observes_dst,calling_code,flag_path,anthem_path
0,Afghanistan,https://en.wikipedia.org/wiki/Afghanistan,Kabul 34°31′N 69°11′E ﻿ / ﻿ 34.517°N 69.183°E ...,Pashto Dari,Dharmic,"652,864 km 2 (252,072 sq mi) ( 40th )",35–50 million ( 36th ),NaN,NaN,Afghani ( افغانى ) ( AFN ),UTC +4:30 Lunar Hijri calendar ( Afghanistan...,0,NaN,countries/Afghanistan_flag.png,NaN
12,Bangladesh,https://en.wikipedia.org/wiki/Bangladesh,Dhaka 23°45′50″N 90°23′20″E ﻿ / ﻿ 23.76389°N 9...,Bengali,Dharmic,"148,460 km 2 (57,320 sq mi) ( 93rd )","173,562,364 ( 8th )",NaN,NaN,Taka ( ৳ ) ( BDT ),UTC +6 ( BST ),0,+880,countries/Bangladesh_flag.png,countries/Bangladesh_anthem.oga
18,Bhutan,https://en.wikipedia.org/wiki/Bhutan,Thimphu 27°28.0′N 89°38.5′E ﻿ / ﻿ 27.4667°N 89...,Dzongkha,Dharmic,"38,394 km 2 (14,824 sq mi) ( 133rd )","777,486 ( 159th )",NaN,NaN,Ngultrum (BTN) Indian rupee (₹),UTC +06 ( BTT ),0,+975,countries/Bhutan_flag.png,NaN
73,India,https://en.wikipedia.org/wiki/India,New Delhi 28°36′50″N 77°12′30″E ﻿ / ﻿ 28.61389...,Hindi English,Dharmic,"3,287,263 km 2 (1,269,219 sq mi) ( 7th )","1,428,627,663 ( 1st )",NaN,NaN,Indian rupee Sign History Historical Forex Dig...,UTC +05:30 ( IST ),0,+91,countries/India_flag.png,countries/India_anthem.ogg
101,Maldives,https://en.wikipedia.org/wiki/Maldives,Malé 4°10′31″N 73°30′32″E ﻿ / ﻿ 4.17528°N 73.5...,Dhivehi,Dharmic,298 km 2 (115 sq mi) ( 187th ),"601,269",NaN,NaN,Maldivian rufiyaa ( MVR ),UTC +5 ( MVT ),0,+960,countries/Maldives_flag.png,countries/Maldives_anthem.ogg
117,Nepal,https://en.wikipedia.org/wiki/Nepal,Kathmandu 28°10′N 84°15′E ﻿ / ﻿ 28.167°N 84.2...,All mother-tongues (see Languages of Nepal ),Dharmic,"147,516 km 2 (56,956 sq mi) ( 93rd )","31,122,387 ( 49th )",NaN,NaN,Nepalese rupee (रू) (NPR),UTC +05:45 ( Nepal Standard Time ),0,+977,countries/Nepal_flag.png,countries/Nepal_anthem.ogg
127,Pakistan,https://en.wikipedia.org/wiki/Pakistan,Islamabad 33°41′30″N 73°3′0″E ﻿ / ﻿ 33.69167°N...,Urdu English,Dharmic,"881,913 km 2 (340,509 sq mi) ( 33rd )","241,499,431 ( 5th )",NaN,NaN,Pakistani rupee (₨) ( PKR ),UTC +5 ( PKT ),0,+92,countries/Pakistan_flag.png,countries/Pakistan_anthem.oga
162,Sri Lanka,https://en.wikipedia.org/wiki/Sri_Lanka,Sri Jayawardenepura Kotte (legislative) Colom...,Sinhala Tamil,Dharmic,"67,240 km 2 (25,960 sq mi) ( 120th )","23,300,000 ( 60th )",NaN,NaN,Sri Lankan rupee ( Rs ) ( LKR ),UTC +5:30 ( SLST ),0,+94,countries/Sri_Lanka_flag.png,countries/Sri_Lanka_anthem.mp3


In [ ]:
df['largest_religion'].value_counts()

largest_religion
Christianity                 100
Islam                         36
Dharmic                        8
Sunni Islam                    7
Buddhism                       7
no religion                    6
Catholicism                    3
See Religion                   2
v                              2
Freedom                        2
Lutheranism                    1
irreligion                     1
Armenian Apostolic Church      1
Ancient religion               1
Islam Christianity             1
Judaism                        1
79.4                           1
Gender                         1
Hinduism                       1
Proto                          1
No religion                    1
Roman Catholicism              1
Baháʼí                         1
Catholic Church                1
Name: count, dtype: int64